In [29]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: C:\\Users\\jaskew\\Documents\\project_repository\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [30]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    display(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups,falsePositives,&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        display(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
    observed_src.drop_duplicates(subset='indicator', inplace=True)
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
    observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)


'Querying owner: HTOC Org'

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,tags.data,associatedGroups.data,hostName,dnsActive,whoisActive,source,address,url,lastFalsePositive,indicator
0,6755399443033945,2025-04-11T17:46:52Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-16T14:54:14Z,4.0,68.0,HTOC—Two US IP Addresses Reported by the USG a...,...,"[{'id': 494539, 'name': 'NIH CTI', 'lastUsed':...","[{'id': 6755399443005186, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.227.166.13
1,6755399443033888,2025-04-11T17:46:51Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-16T14:54:14Z,4.0,65.0,(U//FOUO) Federal law enforcement and the Inte...,...,"[{'id': 267169, 'name': 'APT & Targeted Attack...","[{'id': 6755399443004631, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.177.80.32
2,4439720,2023-10-06T15:16:45Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-16T14:53:55Z,3.0,5.0,IB-23-10092 Phishing Campaign Used Typosquat D...,...,"[{'id': 39804, 'name': 'food and agriculture',...","[{'id': 261835, 'dateAdded': '2023-10-06T14:53...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.82.141.145
3,5629499571090456,2025-10-08T11:23:53Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-16T14:12:39Z,5.0,79.0,"Introduction\nSince early 2025, Group-IB analy...",...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...","[{'id': 6755399465000876, 'dateAdded': '2025-1...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,157.230.9.58
4,6755399467419654,2025-08-13T11:48:32Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-16T13:42:09Z,3.0,71.0,INC9191888,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,125.88.174.211
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
950,4883044,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,3.0,90.0,ACD R&F processed a malspam campaign with a Ne...,...,"[{'id': 390199, 'name': 'hhs-2024090901', 'las...","[{'id': 455233, 'dateAdded': '2024-09-09T11:22...",NaN,NaN,NaN,https://www.shorturl.at/,NaN,www.shorturl.at/,NaN,www.shorturl.at/
951,4476331,2023-11-16T13:43:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-17T23:24:44Z,4.0,85.0,NaN,...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...","[{'id': 291847, 'dateAdded': '2023-12-15T13:16...",NaN,NaN,NaN,http://selligenttier.naylorcampaigns.com,NaN,selligenttier.naylorcampaigns.com,NaN,selligenttier.naylorcampaigns.com
952,4731407,2024-06-24T17:14:00Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-06-24T17:14:40Z,3.0,80.0,NaN,...,"[{'id': 277225, 'name': 'Alert ID: ac38b1ca', ...","[{'id': 347471, 'dateAdded': '2024-06-24T17:13...",NaN,NaN,NaN,https://oklahomaallergy.com/,NaN,oklahomaallergy.com/,NaN,oklahomaallergy.com/
953,4442727,2023-10-06T23:26:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-28T23:23:43Z,3.0,83.0,NaN,...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,NaN,https://google,NaN,google,NaN,google


In [31]:
# Unnest tags.data in observed_src to get a list of each tag per indicator

# Explode tags.data to one row per tag
tags_exploded = (
    observed_src[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# Extract tag name from each tag object
tags_exploded['tag_name'] = tags_exploded['tags.data'].apply(lambda x: x.get('name') if isinstance(x, dict) else None)

# Aggregate all tag names into a list per indicator
tags_per_indicator = (
    tags_exploded.groupby('indicator')['tag_name']
    .apply(lambda x: [t for t in x if t])
    .reset_index()
    .rename(columns={'tag_name': 'tag_list'})
)

# Merge back to observed_src if desired
observed_src_with_tags = observed_src.merge(tags_per_indicator, on='indicator', how='left')

display(observed_src_with_tags[['indicator', 'tag_list']])

,indicator,tag_list
0,64.227.166.13,"[NIH CTI, APT & Targeted Attacks, unauthorized..."
1,31.177.80.32,"[APT & Targeted Attacks, unauthorized access, ..."
2,68.82.141.145,"[food and agriculture, Compromise Accounts: Em..."
3,157.230.9.58,"[DHA Splunk API, Static Kitten, OS Splunk API,..."
4,125.88.174.211,"[DHA Splunk API, OS Splunk API, FDA Splunk API..."
...,...,...
948,www.shorturl.at/,"[hhs-2024090901, FDA Splunk API, Observed, CDC..."
949,selligenttier.naylorcampaigns.com,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp..."
950,oklahomaallergy.com/,"[Alert ID: ac38b1ca, TLP: AMBER, Health-ISAC, ..."
951,google,"[OS Splunk API, HRSA Splunk API, Observed]"


In [32]:
# List of tags to count (case-insensitive, strip whitespace)
tags_of_interest = [
    "Scanning", "DDoS", "Spam", "Phishing", "Cryptojacking",
    "Credential Stuffing", "Ransomware", "Data Theft",
    "Cross Site Scripting Attacks", "SQL Injections"
]

# Normalize tag_list to lowercase for matching
def tag_counter(tag_list, tag):
    if not isinstance(tag_list, list):
        return 0
    return sum(1 for t in tag_list if isinstance(t, str) and t.strip().lower() == tag.lower())

tag_counts = {}
for tag in tags_of_interest:
    tag_counts[tag] = observed_src_with_tags['tag_list'].apply(lambda lst: tag_counter(lst, tag)).sum()
    # Add a column to observed_src with the list of matching "Botnet" tags per indicator
    botnet_tags = set(tags_of_interest)
    def extract_botnet_tags(tag_list):
        if not isinstance(tag_list, list):
            return []
        return [t for t in tag_list if isinstance(t, str) and t.strip().lower() in {b.lower() for b in botnet_tags}]

    # Map indicator to its tag_list for efficient lookup
    indicator_to_tags = dict(zip(observed_src_with_tags['indicator'], observed_src_with_tags['tag_list']))

    observed_src['Botnet'] = observed_src['indicator'].map(lambda ind: extract_botnet_tags(indicator_to_tags.get(ind, [])))
# Display as a DataFrame for readability
pd.DataFrame(list(tag_counts.items()), columns=['Tag', 'Count'])

,Tag,Count
0,Scanning,0
1,DDoS,85
2,Spam,0
3,Phishing,20
4,Cryptojacking,0
5,Credential Stuffing,0
6,Ransomware,6
7,Data Theft,6
8,Cross Site Scripting Attacks,0
9,SQL Injections,0


In [33]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        display("No files found for the specified date range.")
    else:
        display(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            display(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        display(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=1)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_13088\1319479530.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


"Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251016.csv']"

'Loaded data from 1 files.'

In [34]:
import pandas as pd

df = observed_src.copy()

# explode tags.data
tags_exploded = (
    df[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# normalize all fields in tags.data into flat columns
tags_norm = pd.json_normalize(tags_exploded['tags.data'])
tags_norm.columns = [f"tag_{c}" for c in tags_norm.columns]

# re‑attach indicator
tags_expanded = tags_exploded.reset_index(drop=True).join(tags_norm)

# extract partners
tags_expanded['partner'] = tags_expanded['tag_name'].map(
    lambda n: n[:-len(' Splunk API')] if isinstance(n, str) and n.endswith(' Splunk API') else None
)

# aggregate each tag_* field into a list per indicator
tag_fields = list(tags_norm.columns)
tag_agg = (
    tags_expanded
      .groupby('indicator', as_index=False)
      .agg({
          **{f: list for f in tag_fields},
          'partner': lambda x: [p for p in dict.fromkeys(x) if p]
      })
      .rename(columns={'partner':'partners'})
)


# --- CORE AGGREGATION OF OTHER COLUMNS ---

first_cols = [
    'id','dateAdded','ownerId','ownerName','webLink','type','lastModified', 'falsePositives',
    'rating','confidence','description','summary','observations',
    'lastObserved','privateFlag','active','activeLocked','ip',
    'legacyLink','source','address','url'
]
# Remove 'hostName', 'dnsActive', 'whoisActive' from first_cols to avoid KeyError

# Add 'Botnet' column from observed_src if it exists
if 'Botnet' in observed_src.columns:
    df['Botnet'] = observed_src['Botnet']
    first_cols.append('Botnet')

base_agg = (
    df
      .drop(columns=[
          'createdBy.id','createdBy.userName','createdBy.firstName','createdBy.lastName',
          'createdBy.pseudonym','createdBy.owner','xid','eventType','documentDateAdded',
          'documentType','fileSize','fileName','downVoteCount','upVoteCount','type_group',
          'webLink_group','ownerName_group','ownerId_group','dateAdded_group','id_group',
          'platforms.count','tactics.count',
      ], errors='ignore')
      .groupby('indicator', as_index=False)[
          [c for c in first_cols if c in df.columns]
      ]
      .first()
)

# --- MERGE EVERYTHING BACK ---

agg_df = (
    base_agg
      .merge(tag_agg,   on='indicator', how='left')
#      .merge(group_agg, on='indicator', how='left')
)

def clean_list(lst):
    if not isinstance(lst, list):
        return []
    cleaned = []
    for v in lst:
        # drop any null‑like values
        try:
            if pd.isna(v):
                continue
        except Exception:
            pass
        # drop empty strings
        if isinstance(v, str) and v == "":
            continue
        cleaned.append(v)
    return cleaned

def list_to_csv(lst):
    """
    Takes a cleaned list and returns:
      - a comma-separated string of its items, OR
      - an empty string if there are none.
    """
    if not lst:
        return ""
    return ", ".join(str(v) for v in lst)


for col in ['partners', 'group_ids', 'group_names'] + tag_fields:
    if col in agg_df.columns:
        agg_df[col] = agg_df[col].apply(clean_list).apply(list_to_csv)
    
display(agg_df)



,indicator,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,falsePositives,rating,...,tag_id,tag_name,tag_lastUsed,tag_description,tag_techniqueId,tag_tactics.data,tag_tactics.count,tag_platforms.data,tag_platforms.count,partners
0,101.89.174.236,5629499542017370,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-16T02:58:33Z,0,3.0,...,"471298, 35760, 35057, 30479, 23667, 23630, 235...","DHA Splunk API, OS Splunk API, FDA Splunk API,...","2025-10-16T11:49:24Z, 2025-10-16T07:15:29Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS, HHS"
1,102.129.153.43,4583993,2024-04-29T17:11:45Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-16T07:39:44Z,0,3.0,...,"260189, 245130, 214752, 35752, 35057, 30479, 2...","Alert ID: 49ba45c8, TLP: AMBER, Health-ISAC, V...","2024-04-29T00:00:00Z, 2025-09-04T16:49:49Z, 20...",,,,,,,"FDA, CMS, NIH, IHS, HHS, CDC"
2,103.131.128.10,5629499567010832,2025-09-05T17:42:51Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-11T06:08:04Z,0,3.0,...,"23769, 23576, 749","VA CSOC CTS Splunk, Observed, malicious","2025-10-16T13:20:36Z, 2025-10-16T11:49:24Z, 20...",,,,,,,
3,103.133.61.182,6755399460007421,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-15T21:59:05Z,0,4.0,...,"1469320, 1455870, 505200, 471298, 35057, 23769...","INDICATOR NOTICE 25178.1, Mr Hamza Group, T-Su...","2025-07-18T13:37:52Z, 2025-08-26T18:22:50Z, 20...",,,,,,,"DHA, FDA, HRSA, NIH"
4,103.147.246.222,6755399460007699,2025-07-02T12:05:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-15T05:29:28Z,0,4.0,...,"1469320, 1455870, 505200, 471298, 35760, 35057...","INDICATOR NOTICE 25178.1, Mr Hamza Group, T-Su...","2025-07-18T13:37:52Z, 2025-08-26T18:22:50Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
948,www.deepseek.com,5271608,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-10-16T07:02:58Z,0,3.0,...,"471298, 461545, 35760, 35752, 35057, 30770, 30...","DHA Splunk API, DeepSeek, OS Splunk API, VA OI...","2025-10-16T11:49:24Z, 2025-01-31T14:04:11Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS"
949,www.deepseek.com.cdn.cloudflare.net,5271612,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-10-16T06:57:04Z,0,3.0,...,"471298, 461545, 35752, 30479, 23769, 23630, 23...","DHA Splunk API, DeepSeek, VA OIS CSOC CTS Splu...","2025-10-16T11:49:24Z, 2025-01-31T14:04:11Z, 20...",,,,,,,"DHA, CMS, NIH, IHS"
950,www.shorturl.at/,4883044,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,0,3.0,...,"390199, 35057, 23576, 23575, 1502, 26","hhs-2024090901, FDA Splunk API, Observed, CDC ...","2024-09-10T10:48:34Z, 2025-10-14T19:31:45Z, 20...",,,,,,,"FDA, CDC"
951,www.sthda.com,4565837,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-10-15T21:57:13Z,0,4.0,...,"471298, 35760, 35752, 35057, 30770, 30479, 237...","DHA Splunk API, OS Splunk API, VA OIS CSOC CTS...","2025-10-16T11:49:24Z, 2025-10-16T07:15:29Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS"


In [35]:
# ──────────────────────────────────────────────────────────────────────────────
# Clean enrichment: filter unsupported types, call providers separately, summarize
# ──────────────────────────────────────────────────────────────────────────────
import urllib.parse
import pandas as pd

COL_PATH = "data.enrichment.data"   # adjust if API changes

# Determine candidate indicators (use 'indicator' if present, else 'summary')
key_col = 'indicator' if 'indicator' in agg_df.columns else 'summary'

# Keep only types that Shodan/VT commonly support; others will be VT-only
supported_types = {
    'Address','IPv4','IPv6','Host','Domain','URL','File','SHA1','SHA256','MD5'
}

candidates = (
    agg_df[[key_col, 'type']]
      .dropna(subset=[key_col])
      .astype({key_col: str})
      .drop_duplicates(subset=[key_col])
)

indicator_values = candidates[key_col].tolist()

display(f"Enriching {len(indicator_values)} indicators (VT; Shodan where supported)...")

enriched_results = []
failed = []

for _, row in candidates.iterrows():
    value = row[key_col]
    typ   = str(row.get('type', '') or '')

    try:
        encoded_value = urllib.parse.quote(value, safe="")

        # Build provider query: always VT; add Shodan only for supported types
        providers = ["VirusTotalV3"]
        if typ in supported_types and typ not in {"URL","File","SHA1","SHA256","MD5"}:
            # Prefer Shodan for network-y types
            providers.append("Shodan")

        q = urllib.parse.urlencode({"type": providers}, doseq=True)
        enrich_url = f"/v3/indicators/{encoded_value}/enrich?{q}"

        ro = RequestObject()
        ro.set_http_method("POST")
        ro.set_request_uri(enrich_url)
        ro.set_body({})

        resp = tc.api_request(ro)
        # Many provider errors come back with 200 + status in JSON; don't raise on status
        try:
            data = resp.json()
        except Exception:
            data = {"status": getattr(resp, 'status_code', 'n/a'), "raw": getattr(resp, 'text', None)}

        # Attach key for merge
        data[key_col] = value
        enriched_results.append(data)

    except Exception as e:
        failed.append({key_col: value, "type": typ, "error": str(e)})

# If nothing enriched, carry on with base
if not enriched_results:
    display("No enrichment data retrieved.")
    recent_tags = agg_df.copy()
else:
    df_enriched = (
        pd.json_normalize(enriched_results)
          .drop_duplicates(subset=[key_col], keep="last")
    )

    # Merge enrichment block into base
    recent_tags = agg_df.merge(df_enriched, on=key_col, how="left")

    # ── Flatten enrichment payload without creating duplicate base rows ───────
    if COL_PATH in recent_tags.columns:
        exploded = (
            recent_tags[[key_col, COL_PATH]]
            .explode(COL_PATH)
            .dropna(subset=[COL_PATH])
        )

        enrich_flat = pd.json_normalize(exploded[COL_PATH]).add_prefix("enrich_")
        enrich_flat[key_col] = exploded[key_col].values

        # ---- Aggregation helpers ---------------------------------------------
        def _flatten_lists(values):
            out = []
            for v in values:
                if isinstance(v, list):
                    out.extend(v)
                else:
                    out.append(v)
            return out

        def _agg_obj(series: pd.Series):
            vals = [v for v in series.dropna()]
            if not vals:
                return None
            flat = _flatten_lists(vals)
            if all(not isinstance(v, (list, dict)) for v in flat):
                return list(pd.unique(flat))
            return flat

        obj_cols = enrich_flat.select_dtypes("object").columns.difference([key_col])
        num_cols = enrich_flat.columns.difference(obj_cols.union({key_col}))

        agg_dict = {c: _agg_obj for c in obj_cols}
        agg_dict.update({c: "max" for c in num_cols})

        enrich_wide = (
            enrich_flat
              .groupby(key_col, as_index=False)
              .agg(agg_dict)
        )

        recent_tags = (
            recent_tags.drop(columns=[COL_PATH], errors="ignore")
                       .drop_duplicates(subset=[key_col])
                       .merge(enrich_wide, on=key_col, how="left")
        )

    # Minimal success message
    display(f"Enrichment complete for {recent_tags[key_col].notna().sum()} indicators.")

# Compact failure summary without flooding output
if failed:
    fail_df = pd.DataFrame(failed)
    display(f"{len(failed)} indicators failed enrichment (showing up to 10):")
    display(fail_df.head(10))

# Show preview of key enrichment columns if present
preview_cols = [c for c in [key_col, 'enrich_hostNames', 'enrich_domains', 'enrich_tags', 'enrich_vtMaliciousCount'] if c in recent_tags.columns]

recent_tags.drop(columns=['tag_id', 'tag_lastUsed', 'tag_lastModified', 'tag_ownerId', 
                          'tag_ownerName', 'tag_dateAdded', 'tag_description','tag_tactics.count', 
                          'tag_platform.data', 'tag_platform.count', 'data.id', 'data.dateAdded', 'data.ownerId',
                          'data.webLink', 'data.ownerName', 'data.lastModified', 'data.summary', 'data.ip',
                          'data.legacyLink','data.source', 'enrich_cloudProvider', 'enrich_cloudRegion', 'enrich_type',
                          'id'], inplace=True, errors='ignore')
                          
display(recent_tags[preview_cols].head(20))


'Enriching 953 indicators (VT; Shodan where supported)...'

Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL accentcare.com cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"More than one indicator matches the criteria you specified. Try looking up by ID instead.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host api-docs.deepseek.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host api.deepseek.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host app.hive.co cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response:

'Enrichment complete for 953 indicators.'

'64 indicators failed enrichment (showing up to 10):'

,indicator,type,error
0,accentcare.com,Stripped URL,"b'{""errCode"":""0x1001"",""message"":""The Stripped ..."
1,ajxx98.online,Stripped URL,"b'{""errCode"":""0x1001"",""message"":""More than one..."
2,api-docs.deepseek.com,Host,"b'{""errCode"":""0x1001"",""message"":""The Host api-..."
3,api.deepseek.com,Host,"b'{""errCode"":""0x1001"",""message"":""The Host api...."
4,app.hive.co,Host,"b'{""errCode"":""0x1001"",""message"":""The Host app...."
5,arbourvalley.com,Host,"b'{""errCode"":""0x1001"",""message"":""The Host arbo..."
6,arpdcresources.ca,Host,"b'{""errCode"":""0x1001"",""message"":""The Host arpd..."
7,b.pdf-fast.com,Host,"b'{""errCode"":""0x1001"",""message"":""The Host b.pd..."
8,bimbinlombardia.com,Host,"b'{""errCode"":""0x1001"",""message"":""The Host bimb..."
9,cdcfoundation.org,Host,"b'{""errCode"":""0x1001"",""message"":""The Host cdcf..."


,indicator,enrich_hostNames,enrich_domains,enrich_tags,enrich_vtMaliciousCount
0,101.89.174.236,None,None,[database],8.0
1,102.129.153.43,None,None,None,2.0
2,103.131.128.10,[keralavisionisp-dynamic-10.128.131.103.kerala...,[keralavisionisp.com],None,1.0
3,103.133.61.182,None,None,None,2.0
4,103.147.246.222,[ip-246-222.plb.net.id],[plb.net.id],[vpn],2.0
5,103.150.218.78,[PROXINET-INTERNET-103-150-218-78.proxinet.co.id],[proxinet.co.id],[vpn],1.0
6,103.153.149.26,None,None,None,0.0
7,103.159.194.209,None,None,None,1.0
8,103.169.132.34,None,None,None,2.0
9,103.171.255.188,None,None,None,1.0


In [36]:
#recent_tags = recent_tags.head(50)
recent_tags.loc[recent_tags['enrich_vtMaliciousCount'].isnull(), 'enrich_vtMaliciousCount'] = 0
display(recent_tags)

,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,falsePositives,rating,confidence,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
0,101.89.174.236,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-16T02:58:33Z,0,3.0,60.0,...,[Shanghai],[China],None,None,[China Telecom (Group)],"[{'transport': 'tcp', 'port': 22, 'data': 'SSH...",[CHINANET SHANGHAI PROVINCE NETWORK],[database],None,8.0
1,102.129.153.43,2024-04-29T17:11:45Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-16T07:39:44Z,0,3.0,6.0,...,[Miami],[United States],None,None,[Cogent Communications],"[{'transport': 'tcp', 'port': 80}, {'transport...",[Internet Utilities Africa (PTY) LTD],None,None,2.0
2,103.131.128.10,2025-09-05T17:42:51Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-11T06:08:04Z,0,3.0,75.0,...,[Thrissur],[India],[keralavisionisp.com],[keralavisionisp-dynamic-10.128.131.103.kerala...,[Kerala Vision Broad Band Private Limited],"[{'transport': 'tcp', 'port': 80, 'product': '...",[Kerala Vision Broad Band Private Limited],None,None,1.0
3,103.133.61.182,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-15T21:59:05Z,0,4.0,59.0,...,[Kedondong],[Indonesia],None,None,[PT Tunas Link Indonesia],"[{'transport': 'tcp', 'port': 80, 'data': 'HTT...",[PT Tunas Link Indonesia],None,None,2.0
4,103.147.246.222,2025-07-02T12:05:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-15T05:29:28Z,0,4.0,59.0,...,[Pasarkemis],[Indonesia],[plb.net.id],[ip-246-222.plb.net.id],[PT. Putra Lebak Banten],"[{'transport': 'tcp', 'port': 53}, {'transport...",[PT. Putra Lebak Banten],[vpn],None,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
948,www.deepseek.com,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-10-16T07:02:58Z,0,3.0,73.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
949,www.deepseek.com.cdn.cloudflare.net,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-10-16T06:57:04Z,0,3.0,71.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
950,www.shorturl.at/,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,0,3.0,90.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
951,www.sthda.com,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-10-15T21:57:13Z,0,4.0,39.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [38]:
# Count how many indicators are associated with a unique enrich_domains value

# Only proceed if 'enrich_domains' exists in exploded DataFrame
if 'enrich_domains' in exploded.columns:
    # Drop rows where enrich_domains is missing or NaN
    domains_df = exploded.dropna(subset=['enrich_domains'])

    # If enrich_domains is a list, flatten to individual domain rows
    def flatten_domains(row):
        val = row['enrich_domains']
        if isinstance(val, list):
            return [(row['indicator'], d) for d in val if pd.notna(d)]
        elif pd.notna(val):
            return [(row['indicator'], val)]
        return []

    flat = []
    for _, row in domains_df.iterrows():
        flat.extend(flatten_domains(row))

    flat_df = pd.DataFrame(flat, columns=['indicator', 'domain'])

    # Count unique indicators per domain
    domain_counts = (
        flat_df.groupby('domain')['indicator']
        .nunique()
        .reset_index()
        .rename(columns={'indicator': 'indicator_count'})
        .sort_values('indicator_count', ascending=False)
    )

    display(domain_counts)
else:
    display("Column 'enrich_domains' does not exist in the provided DataFrame. Skipping domain association count.")

"Column 'enrich_domains' does not exist in the provided DataFrame. Skipping domain association count."

In [39]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=365)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_13088\1921769075.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251016.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251015.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251014.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251013.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251012.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251011.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251010.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251009.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251008.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251007.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251006.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251005.csv', 'Z:/HTOC/Data_Analytics/Data/Op

In [40]:
# Aggregate observed_data_df by 'indicator', counting unique obs_date occurrences
agg_by_indicator = (
    observed_data_df
    .groupby('indicator', as_index=False)['obs_date']
    .nunique()
    .rename(columns={'obs_date': 'obs_days_count'})
)

agg_by_indicator = agg_by_indicator[agg_by_indicator['indicator'].isin(recent_tags['indicator'])]
# Map obs_days_count from agg_by_indicator to recent_tags by indicator, rename to obs_count
recent_tags = recent_tags.merge(
    agg_by_indicator.rename(columns={'obs_days_count': 'obs_count'}),
    on='indicator',
    how='left'
)
display(agg_by_indicator)

,indicator,obs_days_count
19,101.89.174.236,149
24,102.129.153.43,19
87,103.131.128.10,8
92,103.133.61.182,22
101,103.147.246.222,4
...,...,...
5447,www.deepseek.com,241
5448,www.deepseek.com.cdn.cloudflare.net,194
5482,www.shorturl.at/,181
5485,www.sthda.com,250


In [41]:
import re

# helper to flatten list-or-str into a list of strings
def flatten_hosts(hosts):
    if isinstance(hosts, list):
        return [h for h in hosts if isinstance(h, str)]
    elif isinstance(hosts, str):
        return [hosts]
    return []

# regex: 
# ^[A-Za-z]+       → first label is one “word” (letters only)
# (?:\.[^.]+){3}$  → exactly three more “.” followed by non-dot characters
four_label_word_prefix = re.compile(r'^[A-Za-z]+(?:\.[^.]+){3}$')

# Try to match on 'enrich_org' instead of 'enrich_hostNames'
if 'enrich_org' in recent_tags.columns:
    global matched_results
    exploded = recent_tags.explode('enrich_org')

    mask = exploded['enrich_org'].astype(str).str.match(four_label_word_prefix, na=False)

    matched = recent_tags.loc[exploded[mask].index.unique()]
    matched_results = matched[['indicator','enrich_org']]
else:
    matched = pd.DataFrame(columns=recent_tags.columns)
    matched_results = None


display(matched_results)


,indicator,enrich_org


In [42]:
def has_false_positive(tag_str):
    if not isinstance(tag_str, str):
        return False
    tags = [t.strip().lower() for t in tag_str.split(",")]
    return any("false positive" in t for t in tags)

false_positive_indicators = recent_tags[
    recent_tags['tag_name'].apply(has_false_positive)
]

display(false_positive_indicators[['indicator', 'tag_name']])


,indicator,tag_name


In [44]:
import pandas as pd
import numpy as np

df = recent_tags.copy()

# ── Severity Binning ────────────────────────────
scoring_bins = [0, 200, 500, 800, 1001]
label_bins = ['low', 'medium', 'high', 'critical']

# ── Feature Weights / Params ────────────────────
Weights = {
    "MALICIOUS_WEIGHT": 2.00,
    # Interpreted as penalty strength in [0,1]; 0.02 ⇒ up to 2% reduction at 365 obs
    "OBSERVATION_COUNT_WEIGHT": 0.02,
    "CONTINUITY_WEIGHT": 0.10,
    "TC_RATING": 0.05,
    "TC_CONFIDENCE": 0.02,
}

BOTNET_ACTIONS = {
    'scanning','ddos','spam','phishing','cryptojacking','credential stuffing','ransomware'
}

# Known feature maximums (absolute, domain-informed)
VT_MAX = 94
MAX_OBS_REALISTIC = 365
MAX_RATING = 5
MAX_CONFIDENCE = 100

# Policy multipliers
FALSE_POSITIVE_WEIGHT = 0.9         # 10% penalty
BOTNET_MULTIPLIER     = 0.4         # 60% penalty when botnet-related
SCANNER_PENALTY_MULTIPLIER = 0.7    # 30% penalty for scanner/benign-crawler tags
DATA_QUALITY_FLOOR    = 0.85        # at worst 15% reduction for poor completeness

# ── Input caps ──────────────────────────────────
df['obs_count'] = pd.to_numeric(df.get('obs_count', 0), errors='coerce').fillna(0).clip(0, MAX_OBS_REALISTIC)
df['enrich_vtMaliciousCount'] = (
    pd.to_numeric(df.get('enrich_vtMaliciousCount', 0), errors='coerce')
      .fillna(0).clip(0, VT_MAX)
)
df['rating']     = pd.to_numeric(df.get('rating', 0), errors='coerce').fillna(0).clip(0, MAX_RATING)
df['confidence'] = pd.to_numeric(df.get('confidence', 0), errors='coerce').fillna(0).clip(0, MAX_CONFIDENCE)
df['type'] = df.get('type', '').astype(str)

# ── Base additive evidence (no obs additive term) ───────────────
df['malicious_scaled']     = np.log1p(df['enrich_vtMaliciousCount'])
df['malicious_raw_score']  = df['malicious_scaled'] * Weights['MALICIOUS_WEIGHT']
df['continuity_val']       = df['type'].map({
    'Address': 1, 'IPv4': 1, 'IPv6': 1,
    'Domain': 2, 'Host': 2, 'URL': 2,
    'EmailAddress': 2, 'EmailSubject': 2,
    'SHA1': 3, 'SHA256': 3, 'MD5': 3
}).fillna(0)
df['continuity_raw_score'] = df['continuity_val'] * Weights['CONTINUITY_WEIGHT']
df['tc_raw_rating']        = df['rating']     * Weights['TC_RATING']
df['tc_raw_confidence']    = df['confidence'] * Weights['TC_CONFIDENCE']

df['raw_score'] = (
    df['malicious_raw_score'] +
    df['continuity_raw_score'] +
    df['tc_raw_rating'] +
    df['tc_raw_confidence']
)

# ── Observation penalty (multiplier; linear) ────────────────────
OBS_PENALTY_STRENGTH = float(Weights['OBSERVATION_COUNT_WEIGHT'])
OBS_MIN_MULTIPLIER   = 0.50  # floor so max reduction is 50% (tune to taste)

obs_frac = df['obs_count'] / MAX_OBS_REALISTIC
df['obs_penalty_multiplier'] = (1.0 - OBS_PENALTY_STRENGTH * obs_frac).clip(OBS_MIN_MULTIPLIER, 1.0)
df['raw_score'] *= df['obs_penalty_multiplier']

# ── Data quality multiplier (light guard) ───────────────────────
needed = ['type','rating','confidence','enrich_vtMaliciousCount']
present_frac = df[needed].notna().sum(axis=1) / len(needed)
df['data_quality_multiplier'] = present_frac.clip(DATA_QUALITY_FLOOR, 1.0)
df['raw_score'] *= df['data_quality_multiplier']

# ── Botnet flag (robust, multi-word, list/string) ───────────────
col = df.get('Botnet', None)
if col is None:
    df['botnet_flag'] = 0
else:
    def is_botnet(val):
        text = " ".join(map(str, val)).lower() if isinstance(val, (list, set, tuple)) else str(val).lower()
        return int(any(action in text for action in BOTNET_ACTIONS))
    df['botnet_flag'] = pd.Series(col).apply(is_botnet).astype(int)

# ── False Positive penalty (preview + conditional apply) ────────
if 'falsePositives' in df.columns:
    df['falsePositives'] = pd.to_numeric(df['falsePositives'], errors='coerce').fillna(0)
    mask_fp = df['falsePositives'] > 0
    # Preview: always "what if" penalized
    df['false_positive_raw_score'] = df['raw_score'] * FALSE_POSITIVE_WEIGHT
    # Apply FP only where present
    df.loc[mask_fp, 'raw_score'] = df.loc[mask_fp, 'false_positive_raw_score']
else:
    df['false_positive_raw_score'] = df['raw_score']

# ── Botnet penalty application ──────────────────────────────────
df['botnet_penalty_multiplier'] = np.where(df['botnet_flag'] == 1, BOTNET_MULTIPLIER, 1.0)
df['raw_score'] *= df['botnet_penalty_multiplier']

# ── Scanner penalty (robust tag parse) ──────────────────────────
def has_scanner_tag(val):
    scanners = {'scanner','masscan','zmap','shodan','censys'}
    if isinstance(val, (list, set, tuple)):
        t = " ".join(map(str, val)).lower()
    elif isinstance(val, str):
        # supports csv-ish strings
        t = " ".join(x.strip() for x in val.split(',')).lower()
    else:
        t = str(val).lower()
    return any(s in t for s in scanners)

if 'enrich_tags' in df.columns:
    scanner_mask = df['enrich_tags'].apply(has_scanner_tag)
else:
    scanner_mask = pd.Series(False, index=df.index)

df['scanner_penalty_multiplier'] = np.where(scanner_mask, SCANNER_PENALTY_MULTIPLIER, 1.0)
df['raw_score'] *= df['scanner_penalty_multiplier']

# ── Absolute Cap (multipliers are NOT in cap; only additive parts) ───────────
RAW_SCORE_CAP = (
    np.log1p(VT_MAX) * Weights['MALICIOUS_WEIGHT'] +
    float(df['continuity_val'].max() if len(df) else 3) * Weights['CONTINUITY_WEIGHT'] +
    (MAX_RATING * Weights['TC_RATING']) +
    (MAX_CONFIDENCE * Weights['TC_CONFIDENCE'])
)

# ── Normalize to 0–1000 ─────────────────────────────────────────
df['Threat_Score'] = (1000 * (df['raw_score'] / RAW_SCORE_CAP).clip(0, 1)).round().fillna(0).astype(int)

# ── Assign Severity bin ─────────────────────────────────────────
df['Severity'] = pd.cut(df['Threat_Score'], bins=scoring_bins, labels=label_bins, right=False)

# ── Explanation (drivers + multipliers) ─────────────────────────
_NAME_MAP = {
    'malicious_raw_score': 'VT malicious (log-scaled)',
    'continuity_raw_score': 'Continuity (indicator type)',
    'tc_raw_rating': 'TC rating',
    'tc_raw_confidence': 'TC confidence',
}

def build_explanation(row):
    parts = {
        'malicious_raw_score': float(row.get('malicious_raw_score', 0) or 0),
        'continuity_raw_score': float(row.get('continuity_raw_score', 0) or 0),
        'tc_raw_rating': float(row.get('tc_raw_rating', 0) or 0),
        'tc_raw_confidence': float(row.get('tc_raw_confidence', 0) or 0),
    }
    total = float(row.get('raw_score', 0) or 0)
    score = float(row.get('Threat_Score', 0) or 0)
    sev = str(row.get('Severity', 'nan'))

    # Top additive drivers
    contrib = sorted(parts.items(), key=lambda kv: abs(kv[1]), reverse=True)[:3]
    driver_bits = []
    for k, v in contrib:
        label = _NAME_MAP.get(k, k)
        if total != 0:
            pct = 100.0 * (v / total)
            driver_bits.append(f"{label} ({v:+.2f}, {pct:+.1f}% of total)")
        else:
            driver_bits.append(f"{label} ({v:+.2f})")

    # Multipliers
    obs_mult     = float(row.get('obs_penalty_multiplier', 1.0))
    dq_mult      = float(row.get('data_quality_multiplier', 1.0))
    botnet_mult  = float(row.get('botnet_penalty_multiplier', 1.0))
    scanner_mult = float(row.get('scanner_penalty_multiplier', 1.0))
    fp_cnt       = int(row.get('falsePositives', 0) or 0)

    obs_note    = f"Observations ×{obs_mult:.2f} ({(1-obs_mult)*100:.1f}% reduction)."
    dq_note     = f"Data quality ×{dq_mult:.2f}."
    botnet_note = "Botnet ×0.40 applied." if botnet_mult < 1.0 else "No botnet penalty."
    fp_note     = f"{fp_cnt} false positive tag(s): ×0.90 applied." if fp_cnt > 0 else "No false positive tags."
    scan_note   = "Scanner ×0.70 applied." if scanner_mult < 1.0 else "No scanner penalty."

    cap_text = f"Raw score uses {100.0 * (total / RAW_SCORE_CAP):.1f}% of cap; normalized to {score:.0f}/1000." if RAW_SCORE_CAP > 0 else f"Normalized score {score:.0f}/1000."

    return (
        f"Severity: {sev}. Top drivers: " + "; ".join(driver_bits) + ". "
        f"{obs_note} {dq_note} {botnet_note} {fp_note} {scan_note} {cap_text}"
    )

df['Explanation'] = df.apply(build_explanation, axis=1)

# ── De-dupe and show ────────────────────────────────────────────
if 'indicator' in df.columns:
    df.drop_duplicates(subset='indicator', inplace=True)

display(df[['indicator','type','enrich_vtMaliciousCount','obs_count',
            'rating','obs_penalty_multiplier',
            'botnet_flag','falsePositives',
            'Threat_Score','Severity','Explanation']])


,indicator,type,enrich_vtMaliciousCount,obs_count,rating,obs_penalty_multiplier,botnet_flag,falsePositives,Threat_Score,Severity,Explanation
0,101.89.174.236,Address,8.0,149.0,3.0,0.991836,0,0,502,high,Severity: high. Top drivers: VT malicious (log...
1,102.129.153.43,Address,2.0,19.0,3.0,0.998959,0,0,222,medium,Severity: medium. Top drivers: VT malicious (l...
2,103.131.128.10,Address,1.0,8.0,3.0,0.999562,0,0,271,medium,Severity: medium. Top drivers: TC confidence (...
3,103.133.61.182,Address,2.0,22.0,4.0,0.998795,1,0,127,low,Severity: low. Top drivers: VT malicious (log-...
4,103.147.246.222,Address,2.0,4.0,4.0,0.999781,1,0,127,low,Severity: low. Top drivers: VT malicious (log-...
...,...,...,...,...,...,...,...,...,...,...,...
948,www.deepseek.com,Host,0.0,241.0,3.0,0.986795,0,0,155,low,Severity: low. Top drivers: TC confidence (+1....
949,www.deepseek.com.cdn.cloudflare.net,Host,0.0,194.0,3.0,0.989370,0,0,152,low,Severity: low. Top drivers: TC confidence (+1....
950,www.shorturl.at/,Stripped URL,0.0,181.0,3.0,0.990082,0,0,167,low,Severity: low. Top drivers: TC confidence (+1....
951,www.sthda.com,Host,0.0,250.0,4.0,0.986301,0,0,101,low,Severity: low. Top drivers: TC confidence (+0....


# Save the list of indicators from df to the specified CSV file
indicator_list = df['indicator'].drop_duplicates().sort_values()
output_path = r"C:\Users\jaskew\Documents\project_repository\data\indicator library\indicators.csv"
indicator_list.to_csv(output_path, index=False, header=True)
print(f"Saved {len(indicator_list)} indicators to {output_path}")